In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI()

In [ ]:
#define State
class LLMqa(TypedDict):

    title: str
    outline: str
    blog: str
    score: str

In [7]:
def blog_outline(state: LLMqa) -> LLMqa:

    #extract title
    title = state['title']

    #prompt for outline based on title and send to llm
    prompt = f'Write a blog outline based in the title: {title}'
    outline = model.invoke(prompt).content

    #update outline
    state['outline'] = outline

    return state

In [5]:
def blog_content(state: LLMqa) -> LLMqa:

    #extract title and outline
    title = state['title']
    outline = state['outline']

    #prompt and send
    prompt = f'Write a blog based on the title - {title}, and the outline - {outline}'
    blog = model.invoke(prompt).content

    #update blog in state
    state['blog'] = blog

    return state

In [6]:
def blog_evaluation(state: LLMqa) -> LLMqa:

    #exctract title, outline and blog
    title = state['title']
    outline = state['outline']
    blog = state['blog']

    #prompt and send
    prompt = f'Evaluate the blog provided and give it a score out of 10, based on the accuracy towards title - {title} and the outline - {outline}. The blog is - {blog}'
    score = model.invoke(prompt)

    #update
    state['score'] = score

    return state

In [8]:
#create graph
graph = StateGraph(LLMqa)

#add nodes
graph.add_node('blog_outline', blog_outline)
graph.add_node('blog_content', blog_content)
graph.add_node('blog_evaluation', blog_evaluation)

#add edges
graph.add_edge(START, 'blog_outline')
graph.add_edge('blog_outline', 'blog_content')
graph.add_edge('blog_content', 'blog_evaluation')
graph.add_edge('blog_evaluation', END)

#compile the graph
workflow = graph.compile()


In [9]:
#execute the graph
initial_state = {'title': 'Rise of AI in Healthcare.'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in Healthcare.', 'outline': 'I. Introduction\n- Explanation of AI technology\n- Overview of current use of AI in healthcare\n\nII. The Benefits of AI in Healthcare\n- Improved diagnosis accuracy\n- Personalized treatment plans\n- Enhanced patient care and monitoring\n\nIII. Current Applications of AI in Healthcare\n- Electronic health records management\n- Medical imaging analysis\n- Drug discovery and development\n\nIV. Future Potential of AI in Healthcare\n- Predictive analytics for disease prevention\n- Robotic surgery and patient care\n- Virtual healthcare assistants\n\nV. Ethical and Privacy Concerns\n- Data security and protection\n- Bias in AI algorithms\n- Patient consent and autonomy\n\nVI. Challenges and Limitations\n- Cost of implementation\n- Staff training and acceptance\n- Regulatory hurdles\n\nVII. Conclusion\n- Recap of the rise of AI in healthcare\n- Discussion of the potential impact and future developments in the field.', 'blog': 'The Rise of AI

In [12]:
print(final_state['score'])

content='I would give this blog a score of 9 out of 10 for accuracy towards the title and outlined content. The blog effectively covers the rise of AI in healthcare by discussing the current use, benefits, applications, future potential, ethical concerns, challenges, and limitations surrounding AI in the healthcare industry. The content is well-organized and provides a comprehensive overview on the topic. The only suggestion for improvement would be to provide more specific examples or case studies to further illustrate the points made. Overall, it is a well-researched and informative blog on the rise of AI in healthcare.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 986, 'total_tokens': 1103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'ope